# ENTRAÎNEMENT — Détecteur + Lecteur de labelsEntraîne les 2 modèles de labels, **dans l'ordre**, puis sauve leurs checkpoints :- **A. DÉTECTEUR** → `runs_label_detector/run_latest/best_label_detector.pth`- **B. LECTEUR** → `runs_label_reader/run_latest/best_label_reader.pth`> Le modèle **signal** s'entraîne à part (`training_signal.ipynb`).> Une fois entraîné ici, lance **`test.ipynb`** pour la digitalisation.

# ══════════════════════════════════════════════════════════════════════# SECTION A — DÉTECTEUR (entraînement)# ══════════════════════════════════════════════════════════════════════

# Détecteur de labels de dérivation (heatmap de points)

**But** : trouver **où** sont les ~12 étiquettes de dérivation sur une image ECG, quel que soit le layout (3×4, 6×2, +rythme…), la perspective ou la résolution. Le détecteur ne lit pas le texte — il sort une **heatmap class-agnostic** dont les pics = positions des labels. Le **lecteur** (`training_label_reader`) dira ensuite *quelle* dérivation à chaque pic.

**Pourquoi** : sur le réel, placer les crops à la main ne marche pas (layouts variables, perspective → boîtes « à côté de la plaque »). Le détecteur automatise la localisation. GT gratuite : `label_centers` du synthétique.

**Pipeline final** : `image → détecteur (pics) → crops centrés → lecteur → 12 dérivations`.

- Entraînement : `output_augmentation` (031/032 train, 033 val), cible = heatmap gaussienne aux `label_centers`.
- Sortie : 1 canal (présence de label), pics extraits par maximum local.


In [ ]:
# ── Cellule 1 : imports + config + letterbox ──
import os, sys, glob, io, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

PROJECT_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from shared.npz_schema import load_unified

%matplotlib inline

IMG_DIR  = os.path.join(PROJECT_ROOT, "data", "output_augmentation", "images")
LBL_DIR  = os.path.join(PROJECT_ROOT, "data", "output_augmentation", "labels")
MASK_DIR = os.path.join(PROJECT_ROOT, "data", "output_augmentation", "masks")   # mask_all_signals = negatifs durs
REAL_DIR = os.path.join(PROJECT_ROOT, "data", "output_real")
CACHE_DIR= os.path.join(PROJECT_ROOT, "data", "training", "label_detector_cache")
OUT_DIR  = os.path.join(PROJECT_ROOT, "data", "training", "runs_label_detector")
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

IN_W, IN_H = 1024, 736      # entree du U-Net (+ haute resolution -> pics + precis sur les petits labels des membres)
SIGMA = 2.0                 # pic gaussien plus serre -> localisation plus precise
MAXC  = 24                  # nb max de labels caches par image
TRAIN_SOURCES = ["ECG_031", "ECG_032"]
VAL_SOURCES   = ["ECG_033"]
BATCH = 6                   # baisse vs 768 (1024 = ~1.8x VRAM)
EPOCHS = 30
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def letterbox(img, out_w=IN_W, out_h=IN_H, pad=255):
    """Redimensionne en gardant le ratio + pad (centre). Retourne (canvas, s, ox, oy)."""
    h, w = img.shape[:2]
    s = min(out_w / w, out_h / h)
    nw, nh = int(round(w * s)), int(round(h * s))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.full((out_h, out_w, 3), pad, np.uint8)
    ox, oy = (out_w - nw) // 2, (out_h - nh) // 2
    canvas[oy:oy + nh, ox:ox + nw] = resized
    return canvas, s, ox, oy

print("Device:", DEVICE, "| entree", IN_W, "x", IN_H)


In [ ]:
# ── Cellule 2 : cache (images letterboxées + centres en coords d'entrée) ──
def _gauss(sigma):
    r = int(round(3 * sigma)); ax = np.arange(-r, r + 1)
    xx, yy = np.meshgrid(ax, ax)
    return np.exp(-(xx**2 + yy**2) / (2 * sigma**2)).astype(np.float32), r
GK, GR = _gauss(SIGMA)

def build_cache(sources, tag, force=False):
    # cache = images letterboxees + centres + MASQUE SIGNAL letterboxe (negatifs durs)
    key = f"{tag}_{IN_W}x{IN_H}"
    fi = os.path.join(CACHE_DIR, f"{key}_imgs.npy"); fc = os.path.join(CACHE_DIR, f"{key}_cts.npy")
    fs = os.path.join(CACHE_DIR, f"{key}_sig.npy")
    if not force and os.path.exists(fi) and os.path.exists(fc) and os.path.exists(fs):
        imgs = np.load(fi, mmap_mode="r"); cts = np.load(fc); sigs = np.load(fs, mmap_mode="r")
        print(f"[{key}] cache : {len(imgs)} images (+ masques signal)"); return imgs, cts, sigs
    files = [f for f in sorted(os.listdir(IMG_DIR))
             if f.endswith(".webp") and any(f.startswith(p) for p in sources)]
    print(f"[{key}] {len(files)} images...")
    imgs = np.zeros((len(files), IN_H, IN_W, 3), np.uint8)
    cts  = np.full((len(files), MAXC, 2), np.nan, np.float32)
    sigs = np.zeros((len(files), IN_H, IN_W), np.uint8)
    keep = 0
    for fname in files:
        stem = fname[:-5]
        try:
            d = load_unified(os.path.join(LBL_DIR, stem), load_maps=False)
            lc = d.get("label_centers")
            img = np.array(Image.open(os.path.join(IMG_DIR, fname)).convert("RGB"))
        except Exception:
            continue
        if lc is None: continue
        canvas, s, ox, oy = letterbox(img)
        imgs[keep] = canvas
        for i, (x, y) in enumerate(lc[:MAXC]):
            cts[keep, i] = [float(x) * s + ox, float(y) * s + oy]
        # masque signal (mask_all_signals) letterboxe a l'identique -> negatifs durs (tracés)
        mp = os.path.join(MASK_DIR, stem, "mask_all_signals.png")
        if os.path.exists(mp):
            m = np.array(Image.open(mp).convert("L"))
            m3 = np.repeat(m[:, :, None], 3, axis=2)
            mc, _, _, _ = letterbox(m3, pad=0)
            sigs[keep] = (mc[:, :, 0] > 127).astype(np.uint8)
        keep += 1
        if keep % 300 == 0: print(f"   {keep}/{len(files)}")
    imgs, cts, sigs = imgs[:keep], cts[:keep], sigs[:keep]
    np.save(fi, imgs); np.save(fc, cts); np.save(fs, sigs)
    print(f"[{key}] -> {keep} images"); return imgs, cts, sigs

def make_heatmap(centers):
    hm = np.zeros((IN_H, IN_W), np.float32)
    for x, y in centers:
        if not np.isfinite(x): continue
        cx, cy = int(round(x)), int(round(y))
        x0, x1 = max(0, cx - GR), min(IN_W, cx + GR + 1)
        y0, y1 = max(0, cy - GR), min(IN_H, cy + GR + 1)
        if x0 >= x1 or y0 >= y1: continue
        gx0, gy0 = x0 - (cx - GR), y0 - (cy - GR)
        patch = GK[gy0:gy0 + (y1 - y0), gx0:gx0 + (x1 - x0)]
        hm[y0:y1, x0:x1] = np.maximum(hm[y0:y1, x0:x1], patch)
    return hm

train_imgs, train_cts, train_sigs = build_cache(TRAIN_SOURCES, "train")
val_imgs,   val_cts,   val_sigs   = build_cache(VAL_SOURCES,   "val")
print("train:", train_imgs.shape, "| val:", val_imgs.shape,
      "| labels/img moy:", round(float(np.isfinite(train_cts[:,:,0]).sum(1).mean()), 1),
      "| signal moy:", round(float(np.asarray(train_sigs).mean()), 4))


In [ ]:
# ── Cellule 3 : Dataset + DataLoaders (+ aperçu d'un échantillon) ──
class DetDataset(Dataset):
    def __init__(self, imgs, cts, sigs, augment=False):
        self.imgs, self.cts, self.sigs, self.aug = imgs, cts, sigs, augment
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img = np.asarray(self.imgs[i]).astype(np.float32) / 255.0
        if self.aug:  # photometrique seulement (la heatmap reste valide)
            img = np.clip(img * np.random.uniform(0.8, 1.2) + np.random.uniform(-0.05, 0.05), 0, 1)
            if np.random.rand() < 0.3:
                img = np.clip(img + np.random.normal(0, 0.02, img.shape), 0, 1).astype(np.float32)
        hm = make_heatmap(self.cts[i])
        sig = np.asarray(self.sigs[i]).astype(np.float32)   # 1 sur tracé -> negatif dur
        x = torch.from_numpy(img).permute(2, 0, 1)
        return x, torch.from_numpy(hm).unsqueeze(0), torch.from_numpy(sig).unsqueeze(0)

train_ds = DetDataset(train_imgs, train_cts, train_sigs, augment=True)
val_ds   = DetDataset(val_imgs,   val_cts,   val_sigs,   augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)

xb, yb, sb = next(iter(train_loader))
print("batch img", tuple(xb.shape), "heatmap", tuple(yb.shape), "signal", tuple(sb.shape))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(xb[0].permute(1,2,0).numpy()); ax[0].set_title("image (letterbox)"); ax[0].axis("off")
ov = xb[0].permute(1,2,0).numpy().copy()
ov[..., 0] = np.maximum(ov[..., 0], yb[0,0].numpy())   # rouge = heatmap GT (labels)
ov[..., 2] = np.maximum(ov[..., 2], sb[0,0].numpy())   # bleu = signal (negatifs durs)
ax[1].imshow(ov); ax[1].set_title("heatmap GT (rouge) + signal/negatifs (bleu)"); ax[1].axis("off"); plt.show()


In [ ]:
# ── Cellule 4 : modèle (U-Net resnet34, 1 canal heatmap) ──
def build_model():
    try:   # poids imagenet (meilleur depart) ; repli aleatoire si pas de reseau
        return smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=1, activation=None)
    except Exception as e:
        print("imagenet indispo (", e, ") -> init aleatoire")
        return smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1, activation=None)
model = build_model().to(DEVICE)
print("U-Net resnet34 | params:", sum(p.numel() for p in model.parameters()))


In [ ]:
# ── Cellule 5 : entraînement (MSE sur heatmap sigmoïde) ──
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)
# MSE PONDEREE : la heatmap est creuse (99% de zeros). Un MSE simple s'effondre (le modele
# sort ~0 partout). On pondere les pixels-pics x(1+POS_W*cible) pour forcer l'apprentissage.
POS_W = 80.0
NEG_W = 25.0   # NEGATIFS DURS : poids sur les pixels de TRACE (signal) hors-label -> apprend a NE PAS tirer sur les tracés (corrige les FP type QRS / colonne-fantome des longues bandes)
def weighted_mse(pred, tgt, sig=None):
    w = 1.0 + POS_W * tgt
    if sig is not None:
        w = w + NEG_W * sig * (tgt < 0.1).float()   # tracé ET pas de label -> negatif dur
    return (w * (pred - tgt) ** 2).mean()
import datetime, shutil
run_dir = os.path.join(OUT_DIR, "run_latest")
_old = os.path.join(run_dir, "best_label_detector.pth")
if os.path.exists(_old):   # SAUVEGARDE NON DESTRUCTIVE : archive l'ancien best avant ce run
    _arch = os.path.join(OUT_DIR, "archive"); os.makedirs(_arch, exist_ok=True)
    _dst = os.path.join(_arch, f"best_{datetime.datetime.now():%Y%m%d_%H%M%S}.pth")
    shutil.copy2(_old, _dst); print("ancien best archive ->", _dst)
os.makedirs(run_dir, exist_ok=True)
best_val = float("inf")

def run_epoch(loader, train=True):
    model.train(train); tot, loss_sum = 0, 0.0
    torch.set_grad_enabled(train)
    for xb, yb, sb in loader:
        xb, yb, sb = xb.to(DEVICE), yb.to(DEVICE), sb.to(DEVICE)
        pred = torch.sigmoid(model(xb)); loss = weighted_mse(pred, yb, sb)
        if train:
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # anti-divergence
            optimizer.step()
        loss_sum += loss.item() * len(xb); tot += len(xb)
    torch.set_grad_enabled(True)
    return loss_sum / max(tot, 1)

print(f"Entrainement {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(train_loader, True); va = run_epoch(val_loader, False)
    scheduler.step(va); flag = ""
    if va < best_val:
        best_val = va
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_loss": va,
                    "in_w": IN_W, "in_h": IN_H, "sigma": SIGMA},
                   os.path.join(run_dir, "best_label_detector.pth"))
        flag = "  [BEST]"
    print(f"Epoch {epoch:2d}/{EPOCHS} | train {tr:.5f} | val {va:.5f} | "
          f"lr {optimizer.param_groups[0]['lr']:.1e} | {time.time()-t0:.0f}s{flag}")
print(f"\nMeilleure val MSE : {best_val:.5f} -> {os.path.join(run_dir,'best_label_detector.pth')}")


In [ ]:
# ── Cellule 6 : extraction de pics + éval détection (precision/recall) sur la val ──
def find_peaks(hm, thresh=0.3, min_dist=17):
    """Maxima locaux > thresh (NMS par dilatation). Retourne liste de (x,y)."""
    d = cv2.dilate(hm, np.ones((min_dist, min_dist), np.float32))
    mask = (hm >= d) & (hm > thresh)
    ys, xs = np.where(mask)
    return list(zip(xs.tolist(), ys.tolist()))

@torch.no_grad()
def predict_hm(img_tensor):
    return torch.sigmoid(model(img_tensor.unsqueeze(0).to(DEVICE)))[0, 0].cpu().numpy()

# recharge le MEILLEUR checkpoint (l'etat final en memoire peut avoir diverge -> heatmap diffuse)
_best = os.path.join(OUT_DIR, "run_latest", "best_label_detector.pth")
if os.path.exists(_best):
    model.load_state_dict(torch.load(_best, map_location=DEVICE, weights_only=False)["model_state_dict"])
    print("meilleur checkpoint recharge pour l'eval")
model.eval()
# metrique : un pic predit est "bon" s'il tombe a < TOL px d'un centre GT
TOL = 12; TP = FP = FN = 0
for i in range(min(len(val_ds), 200)):
    x, _, _ = val_ds[i]
    hm = predict_hm(x)
    peaks = find_peaks(hm)
    gt = val_cts[i][np.isfinite(val_cts[i][:, 0])]
    used = set()
    for px, py in peaks:
        dists = [np.hypot(px - gx, py - gy) for gx, gy in gt]
        j = int(np.argmin(dists)) if dists else -1
        if j >= 0 and dists[j] < TOL and j not in used: TP += 1; used.add(j)
        else: FP += 1
    FN += len(gt) - len(used)
prec = TP / max(TP + FP, 1); rec = TP / max(TP + FN, 1)
print(f"Detection (val, 200 img, TOL={TOL}px) : precision {prec:.3f} | recall {rec:.3f} | F1 {2*prec*rec/max(prec+rec,1e-9):.3f}")

# visualisation sur 3 images val
fig, axes = plt.subplots(3, 2, figsize=(12, 11))
for r in range(3):
    x, _, _ = val_ds[r]; hm = predict_hm(x); peaks = find_peaks(hm)
    base = x.permute(1, 2, 0).numpy()
    axes[r,0].imshow(hm, cmap="hot"); axes[r,0].set_title(f"heatmap predite ({len(peaks)} pics)"); axes[r,0].axis("off")
    vis = (base*255).astype(np.uint8).copy()
    for px, py in peaks: cv2.circle(vis, (px, py), 6, (255, 0, 0), 2)
    axes[r,1].imshow(vis); axes[r,1].set_title("pics sur image"); axes[r,1].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# ── Cellule 7 : PIPELINE COMPLET sur PM Cardio (détecteur -> lecteur) ──
# Le detecteur localise les labels automatiquement (n'importe quel layout / perspective),
# puis le lecteur lit chaque crop. Plus aucune coordonnee a la main.
%matplotlib inline
from torchvision.models import resnet18

LEADS = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
READER_CKPT = os.path.join(PROJECT_ROOT, "data", "training", "runs_label_reader", "run_latest", "best_label_reader.pth")

det_ck = torch.load(os.path.join(OUT_DIR, "run_latest", "best_label_detector.pth"), map_location=DEVICE, weights_only=False)
model.load_state_dict(det_ck["model_state_dict"]); model.eval()
rd_ck = torch.load(READER_CKPT, map_location=DEVICE, weights_only=False)
reader = resnet18(weights=None, num_classes=12).to(DEVICE); reader.load_state_dict(rd_ck["model_state_dict"]); reader.eval()
print(f"detecteur (val_loss {det_ck['val_loss']:.5f}) + lecteur (val_acc {rd_ck['val_acc']:.3f}) charges")

# --- A AJUSTER ---
REAL_SUBFOLDER = "augmentation_brightness_120"
REAL_INDEX     = 8
THRESH         = 0.7      # seuil de detection des pics (0.7 propre ; monte vers 0.85 sur layouts a longues bandes)
READER_CONF_MIN = 0.90   # garde une detection seulement si le LECTEUR est sur (coupe les crops sur tracé)
# -----------------
rf = sorted(glob.glob(os.path.join(REAL_DIR, REAL_SUBFOLDER, "*")))
rf = [f for f in rf if f.lower().endswith((".jpg",".jpeg",".png",".webp",".bmp"))]
if not rf: raise FileNotFoundError(f"Aucune image dans {REAL_SUBFOLDER}")
img = np.array(Image.open(rf[REAL_INDEX]).convert("RGB")); H, W = img.shape[:2]

# 1) detection (sur image letterboxée) -> pics -> coords image native
canvas, s, ox, oy = letterbox(img)
xin = torch.from_numpy(canvas.astype(np.float32) / 255).permute(2, 0, 1)
hm = predict_hm(xin); peaks = find_peaks(hm, thresh=THRESH)
pts = [((px - ox) / s, (py - oy) / s, float(hm[py, px])) for px, py in peaks]   # +valeur du pic detecteur
print(f"{os.path.basename(rf[REAL_INDEX])} ({W}x{H}) -> {len(pts)} labels detectes")

# 2) lecture de chaque crop (scale-aware, comme le lecteur)
rscale = W / float(rd_ck.get("train_w", 3648)); half = max(8, int(round(rd_ck["crop_half"] * rscale))); R = rd_ck["model_res"]
def read_label(cx, cy):
    cx, cy = int(cx), int(cy); x0,x1 = max(0,cx-half),min(W,cx+half); y0,y1 = max(0,cy-half),min(H,cy+half)
    c = img[y0:y1, x0:x1]
    if c.size == 0: return "?", 0.0
    cc = cv2.resize(c, (R, R), interpolation=cv2.INTER_AREA)
    t = torch.from_numpy(cc.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): p = torch.softmax(reader(t), 1)[0].cpu().numpy()
    return LEADS[int(p.argmax())], float(p.max())
reads_all = [(cx, cy, *read_label(cx, cy), hv) for cx, cy, hv in pts]   # (cx, cy, lead, conf, hm_val)
# FILTRE 1 : un crop sur tracé (pas de texte) donne une confiance lecteur basse -> coupe.
conf_ok = [r for r in reads_all if r[3] >= READER_CONF_MIN]
# FILTRE 2 (dedup) : detections a moins de DEDUP_R px -> on garde le pic DETECTEUR le plus fort.
DEDUP_R = 2 * half
reads = []
for r in sorted(conf_ok, key=lambda r: -r[4]):   # plus fort pic d'abord
    if all((r[0]-k[0])**2 + (r[1]-k[1])**2 > DEDUP_R**2 for k in reads):
        reads.append(r)
print(f"{len(pts)} pics -> {len(conf_ok)} apres conf>={READER_CONF_MIN} -> {len(reads)} apres dedup")

# 3) affichage : retenues (rouge) vs coupees (gris)
kept_xy = {(int(r[0]), int(r[1])) for r in reads}
disp = img.copy()
for cx, cy, lead, conf, hv in reads_all:
    cx, cy = int(cx), int(cy); keep = (cx, cy) in kept_xy
    col = (255, 0, 0) if keep else (150, 150, 150)
    cv2.rectangle(disp, (cx-half, cy-half), (cx+half, cy+half), col, 3 if keep else 1)
    if keep:
        cv2.putText(disp, f"{lead}", (cx-half, cy-half-8), cv2.FONT_HERSHEY_SIMPLEX, 0.9*rscale, (0,140,255), 3)
sp = 1400 / W
plt.figure(figsize=(16, 11)); plt.imshow(cv2.resize(disp, (int(W*sp), int(H*sp)))); plt.axis("off")
plt.title(f"Pipeline detecteur->lecteur : {len(reads)} labels retenus (gris = coupe)"); plt.show()
print("lectures retenues:", [f"{l}({c:.2f})" for _,_,l,c,_ in reads])


In [ ]:
# ── Cellule 5b : CALIBRATION (temperature scaling) — 100% STANDALONE ──
# Ne depend d'AUCUNE autre cellule : definit tout (imports, chemins, LEADS, load_unified).
# Prerequis unique : le checkpoint best_label_reader.pth doit exister (lecteur entraine).
import os, sys, glob as _glob
import numpy as np, cv2
import torch, torch.nn as nn, torch.nn.functional as _F
from PIL import Image
from torchvision.models import resnet18 as _resnet18

_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main"
if _ROOT not in sys.path: sys.path.insert(0, _ROOT)
from shared.npz_schema import load_unified
_DEV = "cuda" if torch.cuda.is_available() else "cpu"
_LEADS = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
_L2I = {l: i for i, l in enumerate(_LEADS)}
_IMG = os.path.join(_ROOT, "data", "output_augmentation", "images")
_LBL = os.path.join(_ROOT, "data", "output_augmentation", "labels")
_VAL = ["ECG_033"]

_best = os.path.join(_ROOT, "data", "training", "runs_label_reader", "run_latest", "best_label_reader.pth")
_ck = torch.load(_best, map_location=_DEV, weights_only=False)
_reader = _resnet18(weights=None, num_classes=len(_LEADS)).to(_DEV)
_reader.load_state_dict(_ck["model_state_dict"]); _reader.eval()
_H = _ck.get("crop_half", 44); _R = _ck.get("model_res", 160)

_files = [f for f in sorted(_glob.glob(os.path.join(_IMG, "*.webp")))
          if any(os.path.basename(f).startswith(s) for s in _VAL)][:200]
_Xc, _yc = [], []
for _p in _files:
    _stem = os.path.splitext(os.path.basename(_p))[0]
    _d = load_unified(os.path.join(_LBL, _stem))
    _lc, _ln = _d.get("label_centers"), _d.get("label_names")
    if _lc is None or _ln is None: continue
    _lc = np.asarray(_lc); _ln = [str(x) for x in _ln]
    _img = np.array(Image.open(_p).convert("RGB"))
    for _i in range(min(len(_lc), len(_ln))):
        if _ln[_i] not in _L2I: continue
        _cx, _cy = int(_lc[_i][0]), int(_lc[_i][1])
        _crop = _img[max(0, _cy-_H):_cy+_H, max(0, _cx-_H):_cx+_H]
        if _crop.size == 0: continue
        _crop = cv2.resize(_crop, (_R, _R), interpolation=cv2.INTER_AREA)
        _Xc.append(_crop.astype(np.float32)/255.0); _yc.append(_L2I[_ln[_i]])
print(f"{len(_Xc)} crops de validation extraits (res {_R}px)")

_logits = []
with torch.no_grad():
    for _i in range(0, len(_Xc), 256):
        _b = torch.from_numpy(np.stack(_Xc[_i:_i+256])).permute(0, 3, 1, 2).to(_DEV)
        _logits.append(_reader(_b).cpu())
_logits = torch.cat(_logits); _labels = torch.tensor(_yc)

def _ece(probs, labels, n_bins=15):
    conf, pred = probs.max(1); acc = (pred == labels).float(); e = 0.0
    for b in range(n_bins):
        m = (conf > b/n_bins) & (conf <= (b+1)/n_bins)
        if m.sum() > 0:
            e += (m.float().mean() * (acc[m].mean() - conf[m].mean()).abs()).item()
    return e

_ece0 = _ece(_F.softmax(_logits, 1), _labels)
_Tp = torch.nn.Parameter(torch.ones(1)); _opt = torch.optim.LBFGS([_Tp], lr=0.05, max_iter=100)
_nll = nn.CrossEntropyLoss()
def _closure():
    _opt.zero_grad(); l = _nll(_logits/_Tp.clamp(min=1e-2), _labels); l.backward(); return l
_opt.step(_closure)
T = max(float(_Tp.detach()), 1e-2)
_ece1 = _ece(_F.softmax(_logits/T, 1), _labels)
_acc = (_logits.argmax(1) == _labels).float().mean().item()
_ck["temperature"] = T; torch.save(_ck, _best)
print(f"Temperature T = {T:.3f}  (T>1 = confiance adoucie)")
print(f"ECE (calibration) : {_ece0:.4f} -> {_ece1:.4f}   |   accuracy {_acc:.3f} (inchangee par T)")
print(f"T sauve dans {os.path.basename(_best)} -> le test l'appliquera automatiquement")

# ══════════════════════════════════════════════════════════════════════# SECTION B — LECTEUR (entraînement)# ══════════════════════════════════════════════════════════════════════

# Lecteur de labels de dérivation (CNN)

**But** : lire l'**étiquette imprimée** d'une dérivation (« I », « V1 », « aVR »…) sur un petit crop, pour identifier chaque tracé **par son label** (pas par sa position ni sa forme d'onde).

**Pourquoi** : l'identité par label est **immunisée contre le surapprentissage** (on n'a que ~2 ECG sources). Le texte est varié (polices, nomenclatures, augmentations) → ça généralise, y compris au réel.

- **Entraînement** : pipeline synthétique (`output_augmentation`) — GT gratuite (`label_centers` + `label_names`). Train = ECG_031/032, Val = ECG_033.
- **Test** : PM Cardio réel (`output_real`) — qualitatif (pas de GT de position).


In [ ]:
# ── Cellule 1 : imports + config ──
import os, sys, glob, io, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A

PROJECT_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from shared.npz_schema import load_unified

%matplotlib inline

IMG_DIR  = os.path.join(PROJECT_ROOT, "data", "output_augmentation", "images")
LBL_DIR  = os.path.join(PROJECT_ROOT, "data", "output_augmentation", "labels")
REAL_DIR = os.path.join(PROJECT_ROOT, "data", "output_real")
CACHE_DIR= os.path.join(PROJECT_ROOT, "data", "training", "label_reader_cache")
OUT_DIR  = os.path.join(PROJECT_ROOT, "data", "training", "runs_label_reader")
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
LEAD2IDX = {l: i for i, l in enumerate(LEADS)}

CROP_HALF = 44      # demi-cote du crop (px image native) -- assez large pour ne PAS tronquer "III"
MODEL_RES = 160     # taille d'entree du CNN -- 160 (vs 96) : crops plus resolus -> distingue mieux V1/V4, II/I
TRAIN_W   = 3648    # largeur de reference des images d'entrainement -> sert a scaler le crop a l'inference reelle
TRAIN_SOURCES = ["ECG_031", "ECG_032"]
VAL_SOURCES   = ["ECG_033"]

BATCH = 128
EPOCHS = 25
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| classes:", len(LEADS))


In [ ]:
# ── Cellule 2 : extraction des crops de labels -> cache .npy (a lancer une fois) ──
def extract_crops(sources, tag, force=False):
    """Pour chaque image des `sources`, crope les 12 labels (label_centers) et
    enregistre X (N,RES,RES,3) uint8 + y (N,) dans le cache."""
    # le tag de cache encode crop_half + resolution -> changer CROP_HALF/MODEL_RES regenere
    # automatiquement (pas de melange de tailles entre runs).
    key = f"{tag}_h{CROP_HALF}_r{MODEL_RES}"
    fx = os.path.join(CACHE_DIR, f"{key}_X.npy"); fy = os.path.join(CACHE_DIR, f"{key}_y.npy")
    if not force and os.path.exists(fx) and os.path.exists(fy):
        X, y = np.load(fx), np.load(fy)
        print(f"[{key}] cache existant : {len(y)} crops")
        return X, y
    X, y = [], []
    files = [f for f in sorted(os.listdir(IMG_DIR))
             if f.endswith(".webp") and any(f.startswith(p) for p in sources)]
    print(f"[{tag}] {len(files)} images...")
    for n, fname in enumerate(files):
        stem = fname[:-5]
        try:
            d = load_unified(os.path.join(LBL_DIR, stem), load_maps=False)
            lc = d.get("label_centers"); ln = d.get("label_names")
            if lc is None or ln is None:
                continue
            img = np.array(Image.open(os.path.join(IMG_DIR, fname)).convert("RGB"))
        except Exception:
            continue
        H, W = img.shape[:2]
        for i in range(len(lc)):
            lead = str(ln[i])
            if lead not in LEAD2IDX:
                continue
            cx, cy = int(round(float(lc[i][0]))), int(round(float(lc[i][1])))
            x0, x1 = max(0, cx - CROP_HALF), min(W, cx + CROP_HALF)
            y0, y1 = max(0, cy - CROP_HALF), min(H, cy + CROP_HALF)
            crop = img[y0:y1, x0:x1]
            if crop.size == 0:
                continue
            crop = cv2.resize(crop, (MODEL_RES, MODEL_RES), interpolation=cv2.INTER_AREA)
            X.append(crop); y.append(LEAD2IDX[lead])
        if (n + 1) % 300 == 0:
            print(f"   {n+1}/{len(files)} images, {len(y)} crops")
    X = np.asarray(X, np.uint8); y = np.asarray(y, np.int64)
    np.save(fx, X); np.save(fy, y)
    print(f"[{key}] -> {len(y)} crops sauves")
    return X, y

train_X, train_y = extract_crops(TRAIN_SOURCES, "train")
val_X,   val_y   = extract_crops(VAL_SOURCES,   "val")
print("train:", train_X.shape, "| val:", val_X.shape)
print("repartition train par classe:", {LEADS[i]: int((train_y==i).sum()) for i in range(len(LEADS))})


In [ ]:
# ── Cellule 3 : Dataset + DataLoaders ──
class LabelCropDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X, self.y = X, y
        # PAS de flip (le texte ne se reflechit pas). Rotation FAIBLE (6deg) : au-dela,
        # les barres verticales de I/II/III deviennent des traits obliques ambigus.
        self.tf = A.Compose([
            A.Rotate(limit=6, border_mode=cv2.BORDER_REPLICATE, p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.GaussNoise(p=0.2),
        ]) if augment else None

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        img = self.X[i]
        if self.tf is not None:
            img = self.tf(image=img)["image"]
        t = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1)
        return t, int(self.y[i])

train_ds = LabelCropDataset(train_X, train_y, augment=True)
val_ds   = LabelCropDataset(val_X,   val_y,   augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
print("train batches:", len(train_loader), "| val batches:", len(val_loader))


In [ ]:
# ── Cellule 4 : modele (resnet18, 12 classes) ──
# resnet18 (sans pre-entrainement) discrimine bien mieux les glyphes proches
# (I/II/III = compter des barres ; aVR/aVL/aVF = derniere lettre) que le petit CNN.
from torchvision.models import resnet18

def build_model(n_classes=len(LEADS)):
    m = resnet18(weights=None, num_classes=n_classes)
    return m

model = build_model().to(DEVICE)
print("Backbone: resnet18 | parametres:", sum(p.numel() for p in model.parameters()))


In [ ]:
# ── Cellule 5 : entrainement ──
import datetime
# pondération de classes : III/aVR/aVL/aVF sont structurellement plus rares
# (absentes des formats 4x2 et 6x1;6x1) -> on compense pour une accuracy/derivation equitable.
counts = np.bincount(train_y, minlength=len(LEADS)).astype(np.float32)
weights = (counts.sum() / (len(LEADS) * np.clip(counts, 1, None)))
class_w = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
print("poids de classe:", {LEADS[i]: round(float(weights[i]), 2) for i in range(len(LEADS))})
criterion = nn.CrossEntropyLoss(weight=class_w)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)

run_dir = os.path.join(OUT_DIR, "run_latest"); os.makedirs(run_dir, exist_ok=True)
best_acc, hist = 0.0, {"train_acc": [], "val_acc": [], "val_loss": []}

def run_epoch(loader, train=True):
    model.train(train)
    tot, correct, loss_sum = 0, 0, 0.0
    torch.set_grad_enabled(train)
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb); loss = criterion(logits, yb)
        if train:
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_sum += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item(); tot += len(yb)
    torch.set_grad_enabled(True)
    if tot == 0:   # loader vide -> evite ZeroDivisionError (signale plutot un cache/extraction rate)
        return float("nan"), 0.0
    return loss_sum / tot, correct / tot

print(f"Entrainement {EPOCHS} epochs sur {len(train_y)} crops...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, True)
    va_loss, va_acc = run_epoch(val_loader, False)
    scheduler.step(va_acc)
    hist["train_acc"].append(tr_acc); hist["val_acc"].append(va_acc); hist["val_loss"].append(va_loss)
    flag = ""
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_acc": va_acc,
                    "leads": LEADS, "crop_half": CROP_HALF, "model_res": MODEL_RES, "train_w": TRAIN_W},
                   os.path.join(run_dir, "best_label_reader.pth"))
        flag = "  [BEST]"
    print(f"Epoch {epoch:2d}/{EPOCHS} | train acc {tr_acc:.3f} | val acc {va_acc:.3f} "
          f"loss {va_loss:.3f} | lr {optimizer.param_groups[0]['lr']:.1e} | {time.time()-t0:.0f}s{flag}")

print(f"\nMeilleure val accuracy : {best_acc:.4f}  -> {os.path.join(run_dir,'best_label_reader.pth')}")


In [ ]:
# ── Cellule 6 : evaluation sur la val synthetique (matrice de confusion + acc/classe) ──
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        all_pred.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy()); all_true.append(yb.numpy())
all_pred = np.concatenate(all_pred); all_true = np.concatenate(all_true)

acc = (all_pred == all_true).mean()
print(f"Val accuracy globale : {acc:.4f}\n")
print("Accuracy par derivation :")
for i, l in enumerate(LEADS):
    m = all_true == i
    if m.sum(): print(f"   {l:4s}: {(all_pred[m]==i).mean():.3f}  (n={int(m.sum())})")

# matrice de confusion
C = np.zeros((len(LEADS), len(LEADS)), int)
for t, p in zip(all_true, all_pred): C[t, p] += 1
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(C, cmap="Blues")
ax.set_xticks(range(len(LEADS))); ax.set_xticklabels(LEADS, rotation=45)
ax.set_yticks(range(len(LEADS))); ax.set_yticklabels(LEADS)
ax.set_xlabel("Predit"); ax.set_ylabel("Vrai"); ax.set_title(f"Matrice de confusion (val) - acc {acc:.3f}")
for i in range(len(LEADS)):
    for j in range(len(LEADS)):
        if C[i, j]: ax.text(j, i, C[i, j], ha="center", va="center",
                            color="white" if C[i, j] > C.max()*0.5 else "black", fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()


In [ ]:
# ── Cellule 7 : TEST sur PM Cardio (output_real) -- crop SCALE-AWARE, ciblage MANUEL ──
# Correction clef : le crop est mis a l'ECHELLE de l'image. Les images PM Cardio font
# 578..7483 px de large (entrainement = 3648 px). Sans mise a l'echelle, un crop fixe ne
# capte qu'un bout de caractere sur les grandes images -> predictions aleatoires.
#
# Le ciblage des labels est VOLONTAIREMENT manuel : la detection auto sur photo reelle
# n'est pas fiable (labels tres pales, layouts variables 3x4 / 6x2 / +rythme, faux positifs
# des traces/QR/fond -- teste et rejete). Deux etapes :
#   MODE="locate" -> affiche l'image + grille de coordonnees pour LIRE les (x,y) des labels
#   MODE="read"   -> remplis LABEL_CENTERS, le CNN lit chaque crop
# Astuce : pour une photo gondolee/perspective, dewarpe d'abord (image plate = coords nettes).
%matplotlib inline

ckpt = torch.load(os.path.join(OUT_DIR, "run_latest", "best_label_reader.pth"),
                  map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()
_T = ckpt.get("temperature", 1.0)   # calibration
TRAIN_W_CK = ckpt.get("train_w", 3648)
print(f"Modele (val_acc {ckpt['val_acc']:.3f}) | crop_half={ckpt['crop_half']} res={ckpt['model_res']} train_w={TRAIN_W_CK}")

# ----------------------- A AJUSTER -----------------------
REAL_SUBFOLDER = "augmentation_brightness_120"   # photo nette ; sinon digital_data_opacity_015, photos_bents...
REAL_INDEX     = 8             # img_17_page_0 (exemple pre-rempli ci-dessous)
MODE           = "read"        # "locate" : reperer les (x,y) ; "read" : lire LABEL_CENTERS
# Exemple pre-rempli pour img_17. Pour une AUTRE image : mets MODE="locate", relis les (x,y), recolle.
LABEL_CENTERS  = [(505,1190),(505,1535),(505,1880),     # I, II, III
                  (1368,1310),(1368,1555),(1368,1880),  # aVR, aVL, aVF
                  (2186,1235),(2186,1555),(2186,1880),   # V1, V2, V3
                  (3080,1310),(3080,1585),(3080,1890)]   # V4, V5, V6 (bord droit perspectif -> moins sur)
EXPECTED       = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
# ---------------------------------------------------------

real_files = sorted(glob.glob(os.path.join(REAL_DIR, REAL_SUBFOLDER, "*")))
real_files = [f for f in real_files if f.lower().endswith((".jpg",".jpeg",".png",".webp",".bmp"))]
if not real_files:
    raise FileNotFoundError(f"Aucune image dans {os.path.join(REAL_DIR, REAL_SUBFOLDER)} -- verifie REAL_SUBFOLDER")
img = np.array(Image.open(real_files[REAL_INDEX]).convert("RGB"))
H, W = img.shape[:2]
scale = W / float(TRAIN_W_CK)
half  = max(8, int(round(ckpt["crop_half"] * scale)))     # <-- crop a l'echelle de l'image
print(f"Image: {os.path.basename(real_files[REAL_INDEX])} ({W}x{H}) | echelle x{scale:.2f} -> crop {2*half}px (train {2*ckpt['crop_half']}px)")

def predict_crop(img, cx, cy):
    x0, x1 = max(0, cx-half), min(W, cx+half); y0, y1 = max(0, cy-half), min(H, cy+half)
    crop = img[y0:y1, x0:x1]
    if crop.size == 0:   # (cx,cy) hors image -> evite le crash cv2.resize, renvoie un placeholder
        return "?", 0.0, np.zeros((ckpt["model_res"], ckpt["model_res"], 3), np.uint8)
    cc = cv2.resize(crop, (ckpt["model_res"], ckpt["model_res"]), interpolation=cv2.INTER_AREA)
    t = torch.from_numpy(cc.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.softmax(model(t) / _T, 1)[0].cpu().numpy()
    k = int(prob.argmax())
    return LEADS[k], float(prob[k]), crop

if MODE == "locate" or not LABEL_CENTERS:
    # Affiche l'image + grille de coords. Lis les (x,y) des labels, mets-les dans
    # LABEL_CENTERS, passe MODE="read". Le carre rouge central montre la TAILLE du crop.
    step = int(round(200 * scale))
    fig, ax = plt.subplots(figsize=(16, 11)); ax.imshow(img)
    ax.add_patch(plt.Rectangle((W//2-half, H//2-half), 2*half, 2*half, fill=False, ec="red", lw=2))
    ax.set_xticks(np.arange(0, W, step)); ax.set_yticks(np.arange(0, H, step))
    ax.tick_params(labelsize=7); ax.grid(alpha=0.3)
    ax.set_title("MODE locate : lis les (x,y) des labels -> LABEL_CENTERS, puis MODE='read'. Carre rouge = taille du crop.")
    plt.show()
    print("Repere les coordonnees ci-dessus, remplis LABEL_CENTERS et mets MODE='read'.")
else:
    centers  = LABEL_CENTERS
    expected = EXPECTED if len(EXPECTED) == len(centers) else [None]*len(centers)

    # (a) apercu : positions + boites de crop sur l'image
    disp = img.copy(); r = max(6, int(0.004*W))
    for (cx, cy), exp in zip(centers, expected):
        cv2.rectangle(disp, (cx-half, cy-half), (cx+half, cy+half), (255, 0, 0), max(2, r//3))
        if exp: cv2.putText(disp, exp, (cx-half, cy-half-8), cv2.FONT_HERSHEY_SIMPLEX, 0.9*scale, (255,0,0), 2)
    fig, ax = plt.subplots(figsize=(15, 11)); ax.imshow(disp); ax.axis("off")
    ax.set_title("Boites de crop (rouge). Si decalees, corrige LABEL_CENTERS.")
    plt.show()

    # (b) lecture des crops + comparaison a l'attendu (vert=ok, rouge=faux)
    n = len(centers); cols = min(6, n); rows = (n + cols - 1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(2.2*cols, 2.6*rows)); axes = np.array(axes).reshape(-1)
    ok = 0
    for ax, (cx, cy), exp in zip(axes, centers, expected):
        lead, conf, crop = predict_crop(img, cx, cy)
        match = (exp is None) or (lead == exp); ok += int(exp is not None and lead == exp)
        ax.imshow(crop); ax.axis("off")
        ax.set_title(f"{lead} ({conf:.2f})" + (f"\n[att: {exp}]" if exp else ""),
                     fontsize=10, color=("green" if match else "red"))
    for ax in axes[n:]: ax.axis("off")
    if any(e is not None for e in expected):
        plt.suptitle(f"PM Cardio {os.path.basename(real_files[REAL_INDEX])} -- corrects : {ok}/{n}", fontsize=13)
    plt.tight_layout(); plt.show()
